In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.base import clone
from sklearn.model_selection import ParameterGrid

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score
)

### 데이터 불러오기

In [2]:
# 파일 불러오기 
target = "Risk_Label"

# Load datasets
train_df = pd.read_csv(r"../../data/processed/ADASYN/data_train_adasyn.csv")
valid_df = pd.read_csv(r"../../data/processed/ADASYN/data_valid.csv")
test_df = pd.read_csv(r"../../data/processed/ADASYN/data_test.csv")

# Split features/target
X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_valid = valid_df.drop(columns=[target])
y_valid = valid_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]


print("train shape:", X_train.shape, y_train.shape)
print("valid shape:", X_valid.shape, y_valid.shape)
print("test shape:", X_test.shape, y_test.shape)


train shape: (2466, 9) (2466,)
valid shape: (1438, 9) (1438,)
test shape: (822, 9) (822,)


## Grid Search SVM 적합

In [3]:
from sklearn.model_selection import ParameterGrid
from sklearn.base import clone

# SVM pipeline
# 주의: data_train_adasyn / data_valid / data_test가 이미 scaling 완료된 데이터라고 가정
svm_pipeline = Pipeline(
    steps=[("svm", SVC())]
)

# C=0.01이 기존 탐색 범위의 하한이었으므로 더 작은 C까지 확장
C_grid = [0.001, 0.003, 0.01, 0.03, 0.1, 1, 10, 50, 100]
C_grid_poly = [0.001, 0.003, 0.01, 0.03, 0.1, 1, 10, 50]

# Parameter search space
param_grid = [
    {
        "svm__kernel": ["linear"],
        "svm__C": C_grid,
        "svm__class_weight": [None],
    },
    {
        "svm__kernel": ["rbf"],
        "svm__C": C_grid,
        "svm__gamma": ["scale", 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1],
        "svm__class_weight": [None],
    },
    {
        "svm__kernel": ["poly"],
        "svm__C": C_grid_poly,
        "svm__gamma": ["scale", 0.001, 0.003, 0.01, 0.03, 0.1],
        "svm__degree": [2, 3],
        "svm__coef0": [0, 1],
        "svm__class_weight": [None],
    }
]

best_valid_h1 = -1.0
best_valid_g_mean = -1.0
best_valid_recall = -1.0
best_params = None
search_results = []

# Train set으로 학습하고 validation set의 H1 기준으로 best model 선택
# H1 = F1-score와 G-Mean의 조화평균
# H1이 같으면 G-Mean, high-risk recall 순서로 더 높은 모델을 선택
for params in ParameterGrid(param_grid):
    model = clone(svm_pipeline).set_params(**params)
    model.fit(X_train, y_train)

    valid_pred = model.predict(X_valid)

    cm = confusion_matrix(y_valid, valid_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    g_mean = np.sqrt(sensitivity * specificity)

    acc = accuracy_score(y_valid, valid_pred)
    precision = precision_score(y_valid, valid_pred, zero_division=0)
    recall = recall_score(y_valid, valid_pred, zero_division=0)
    f1 = f1_score(y_valid, valid_pred, zero_division=0)
    h1 = 2 * f1 * g_mean / (f1 + g_mean) if (f1 + g_mean) > 0 else 0.0

    search_results.append({
        **params,
        "valid_accuracy": acc,
        "valid_precision": precision,
        "valid_recall": recall,
        "valid_f1": f1,
        "valid_g_mean": g_mean,
        "valid_h1": h1,
    })

    if (
        (h1 > best_valid_h1)
        or (
            np.isclose(h1, best_valid_h1)
            and (
                (g_mean > best_valid_g_mean)
                or (
                    np.isclose(g_mean, best_valid_g_mean)
                    and recall > best_valid_recall
                )
            )
        )
    ):
        best_valid_h1 = h1
        best_valid_g_mean = g_mean
        best_valid_recall = recall
        best_params = params

# Best SVM model 재학습
best_svm = clone(svm_pipeline).set_params(**best_params)
best_svm.fit(X_train, y_train)

# Validation 최종 성능
valid_pred = best_svm.predict(X_valid)

cm_valid = confusion_matrix(y_valid, valid_pred, labels=[0, 1])
tn, fp, fn, tp = cm_valid.ravel()

valid_acc = accuracy_score(y_valid, valid_pred)
valid_precision = precision_score(y_valid, valid_pred, zero_division=0)
valid_recall = recall_score(y_valid, valid_pred, zero_division=0)
valid_f1 = f1_score(y_valid, valid_pred, zero_division=0)

valid_sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
valid_specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
valid_g_mean = np.sqrt(valid_sensitivity * valid_specificity)
valid_h1 = (
    2 * valid_f1 * valid_g_mean / (valid_f1 + valid_g_mean)
    if (valid_f1 + valid_g_mean) > 0
    else 0.0
)

search_results_df = pd.DataFrame(search_results).sort_values(
    ["valid_h1", "valid_g_mean", "valid_recall"],
    ascending=False
)

print("\n=== Best SVM Model by Validation H1 ===")
print("Best Params:", best_params)
print(f"Best Valid H1    : {best_valid_h1:.4f}")
print(f"Best Valid G-Mean: {best_valid_g_mean:.4f}")

print("\n[Validation] Metrics")
print(f"Accuracy  : {valid_acc:.4f}")
print(f"Precision : {valid_precision:.4f}")
print(f"Recall    : {valid_recall:.4f}")
print(f"F1-Score  : {valid_f1:.4f}")
print(f"G-Mean    : {valid_g_mean:.4f}")
print(f"H1        : {valid_h1:.4f}")

print("\n[Validation] Confusion Matrix")
print(cm_valid)

display(search_results_df.head(10))



=== Best SVM Model by Validation H1 ===
Best Params: {'svm__C': 0.1, 'svm__class_weight': None, 'svm__coef0': 0, 'svm__degree': 3, 'svm__gamma': 'scale', 'svm__kernel': 'poly'}
Best Valid H1    : 0.4533
Best Valid G-Mean: 0.5731

[Validation] Metrics
Accuracy  : 0.8679
Precision : 0.4014
Recall    : 0.3519
F1-Score  : 0.3750
G-Mean    : 0.5731
H1        : 0.4533

[Validation] Confusion Matrix
[[1191   85]
 [ 105   57]]


,svm__C,svm__class_weight,svm__kernel,valid_accuracy,valid_precision,valid_recall,valid_f1,valid_g_mean,valid_h1,svm__gamma,svm__coef0,svm__degree
183,0.10,None,poly,0.867872,0.401408,0.351852,0.375000,0.573074,0.453346,scale,0.0,3.0
195,0.10,None,poly,0.869263,0.405797,0.345679,0.373333,0.568739,0.450771,scale,1.0,3.0
177,0.10,None,poly,0.859527,0.373418,0.364198,0.368750,0.579604,0.450737,scale,0.0,2.0
189,0.10,None,poly,0.860918,0.376623,0.358025,0.367089,0.575403,0.448224,scale,1.0,2.0
165,0.03,None,poly,0.874826,0.427419,0.327160,0.370629,0.555838,0.444721,scale,1.0,2.0
70,50.00,None,rbf,0.864395,0.386207,0.345679,0.364821,0.567070,0.443998,0.1,NaN,NaN
225,10.00,None,poly,0.867177,0.395683,0.339506,0.365449,0.563166,0.443259,scale,0.0,2.0
272,50.00,None,poly,0.867177,0.395683,0.339506,0.365449,0.563166,0.443259,0.1,1.0,3.0
153,0.03,None,poly,0.876912,0.436975,0.320988,0.370107,0.551483,0.442947,scale,0.0,2.0
248,10.00,None,poly,0.859527,0.370130,0.351852,0.360759,0.570179,0.441914,0.1,1.0,3.0


### Test 성능 결과 

In [4]:
# Test prediction/evaluation using tuned model
test_pred = best_svm.predict(X_test)

# Test 지표 계산
cm_test = confusion_matrix(y_test, test_pred, labels=[0, 1])
tn, fp, fn, tp = cm_test.ravel()

test_acc = accuracy_score(y_test, test_pred)
test_precision = precision_score(y_test, test_pred, zero_division=0)
test_recall = recall_score(y_test, test_pred, zero_division=0)
test_f1 = f1_score(y_test, test_pred, zero_division=0)
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
test_g_mean = np.sqrt(sensitivity * specificity)
test_h1 = (
    2 * test_f1 * test_g_mean / (test_f1 + test_g_mean)
    if (test_f1 + test_g_mean) > 0
    else 0.0
)

print("[Test] Metrics")
print(f"Accuracy  : {test_acc:.4f}")
print(f"Precision : {test_precision:.4f}")
print(f"Recall    : {test_recall:.4f}")
print(f"F1-Score  : {test_f1:.4f}")
print(f"G-Mean    : {test_g_mean:.4f}")
print(f"H1        : {test_h1:.4f}")
print(f"\n[Test] Confusion Matrix:\n", cm_test)


[Test] Metrics
Accuracy  : 0.8723
Precision : 0.4000
Recall    : 0.3077
F1-Score  : 0.3478
G-Mean    : 0.5385
H1        : 0.4227

[Test] Confusion Matrix:
 [[689  42]
 [ 63  28]]


In [5]:
# =========================
# 마지막 셀. SVM 예측 결과 저장
#    Date 컬럼 + SVM 예측값 컬럼만 저장
# =========================
from pathlib import Path
import pandas as pd
import numpy as np

# 저장 폴더
result_dir = Path(r"../../results/results_ML")
result_dir.mkdir(parents=True, exist_ok=True)

# 현재 test 데이터: test 길이 확인용
test_df = pd.read_csv(r"../../data/processed/ADASYN/data_test.csv")

# Date가 남아 있는 원본/최종 데이터 파일
date_source_path = Path(r"../../data/processed/data_selected.csv")

date_df = pd.read_csv(date_source_path)

# Date 컬럼 확인
if "Date" not in date_df.columns:
    raise ValueError(
        f"{date_source_path} 파일에도 Date 컬럼이 없습니다. "
        "Date가 남아 있는 원본 데이터 파일 경로로 date_source_path를 바꿔야 합니다."
    )

# 날짜순 정렬
date_df["Date"] = pd.to_datetime(date_df["Date"])
date_df = date_df.sort_values("Date").reset_index(drop=True)

# test set은 chronological split 기준 마지막 구간이므로,
# 예측값 개수만큼 마지막 날짜를 가져옴
n_test = len(test_pred)

test_dates = (
    date_df["Date"]
    .tail(n_test)
    .reset_index(drop=True)
    .dt.strftime("%Y-%m-%d")
)

# 길이 확인
if len(test_dates) != len(test_pred):
    raise ValueError(
        f"Date 길이({len(test_dates)})와 예측값 길이({len(test_pred)})가 다릅니다."
    )

# SVM 예측 결과 저장
svm_prediction_result = pd.DataFrame({
    "Date": test_dates,
    "SVM_Pred": np.asarray(test_pred).ravel()
})

save_path = result_dir / "02. SVM_prediction.csv"

svm_prediction_result.to_csv(
    save_path,
    index=False,
    encoding="utf-8-sig"
)

print("SVM 예측 결과 저장 완료")
print(save_path)
print(svm_prediction_result.head())

SVM 예측 결과 저장 완료
..\..\results\results_ML\02. SVM_prediction.csv
         Date  SVM_Pred
0  2022-10-17         0
1  2022-10-18         0
2  2022-10-19         0
3  2022-10-20         0
4  2022-10-21         0
